In [33]:
import polars as pl
import plotly.express as px
from constants import ROUTES, STOPS, DATA_FOLDER, DATA_FILE
import numpy as np
from scipy.spatial import KDTree
from helpers import clean_df
from datetime import date, datetime

In [3]:
df_routes = pl.read_csv(ROUTES)
df_stops = pl.read_csv(STOPS)
df_raw = pl.read_csv(DATA_FOLDER / "sample.CSV", ignore_errors=True)

In [4]:
df_stops.glimpse()

Rows: 19439
Columns: 19
$ StopUID          <str> 'THB267193', 'THB267018', 'THB307459', 'THB300514', 'THB168103', 'THB129966', 'THB129964', 'THB271131', 'THB214833', 'THB290370'
$ StopID           <i64> 267193, 267018, 307459, 300514, 168103, 129966, 129964, 271131, 214833, 290370
$ AuthorityID      <i64> 10, 10, 10, 10, 10, 10, 10, 10, 10, 10
$ StopNameZh_tw    <str> '仁愛新生路口', '光華商場', '光華商場', '臺北車站(鄭州)', '酒泉重慶路口', '酒泉重慶路口', '昌吉重慶路口', '昌吉重慶路口', '捷運大橋頭站', '圓環(南京)'
$ StopNameEn       <str> 'Ren-ai and Xinsheng Intersection', 'Guanghua Computer Market', 'Guanghua Market', 'Taipei Station (Zhengzhou)', 'Jiuquan Chongqing Road Intersection', 'Jiuquan Chongqing Road Intersection', 'Changji Chongqing Road Intersection', 'Changji Chongqing Road Intersection', 'MRT Daqiaotou Station', 'Yuanhuan(Nanking)'
$ PositionLat      <f64> 25.03789, 25.044621, 25.044532, 25.048361, 25.07181, 25.07173, 25.06567, 25.06556, 25.06344, 25.053656
$ PositionLon      <f64> 121.53288, 121.533089, 121.532734, 121.5

In [7]:
df_raw.glimpse()

Rows: 736130
Columns: 26
$ PlateNumb         <str> '393-FL', '393-FL', '805-FZ', '393-FL', '393-FL', '393-FL', '393-FL', '393-FL', '393-FL', '393-FL'
$ OperatorID        <i64> 35, 35, 35, 35, 35, 35, 35, 35, 35, 35
$ OperatorNo        <i64> 1308, 1308, 1308, 1308, 1308, 1308, 1308, 1308, 1308, 1308
$ RouteUID          <str> 'THB0968', 'THB0968', 'THB0968', 'THB0968', 'THB0968', 'THB0968', 'THB0968', 'THB0968', 'THB0968', 'THB0968'
$ RouteID           <i64> 968, 968, 968, 968, 968, 968, 968, 968, 968, 968
$ RouteNameZh_tw    <i64> 968, 968, 968, 968, 968, 968, 968, 968, 968, 968
$ RouteNameEn       <i64> 968, 968, 968, 968, 968, 968, 968, 968, 968, 968
$ SubRouteUID       <str> 'THB096802', 'THB096802', 'THB096802', 'THB096801', 'THB096802', 'THB096801', 'THB096801', 'THB096801', 'THB096802', 'THB096802'
$ SubRouteID        <i64> 96802, 96802, 96802, 96801, 96802, 96801, 96801, 96801, 96802, 96802
$ SubRouteNameZh_tw <i64> 968, 968, 968, 968, 968, 968, 968, 968, 968, 968
$ SubRouteNameE

In [9]:
def assign_nearest_stop(
    df_bus: pl.DataFrame,
    df_stops: pl.DataFrame,
    gps_lat_col: str = "PositionLat",
    gps_lon_col: str = "PositionLon",
    stop_lat_col: str = "PositionLat",
    stop_lon_col: str = "PositionLon",
    stop_name_col: str = "StopNameZh_tw",
    stop_id_col: str = "StopUID",
    out_name_col: str = "NearestStopName",
    out_id_col: str = "NearestStopUID",
    out_dist_col: str = "DistanceToStop_m",
) -> pl.DataFrame:
    """
    For every row in `df_bus`, find the nearest stop in `df_stops` using a
    KDTree built on Cartesian (ECEF) coordinates, and add:
        - `out_name_col`  : StopNameZh_tw of the nearest stop
        - `out_id_col`    : StopUID of the nearest stop
        - `out_dist_col`  : great-circle distance in metres (float, 2 d.p.)

    Converting lat/lon → 3D Cartesian (ECEF) before building the KDTree means
    the Euclidean nearest-neighbour in 3D is also the great-circle nearest-
    neighbour on the sphere — no chunking or haversine loop needed.

    Parameters
    ----------
    df_bus   : GPS ping DataFrame (~736 k rows)
    df_stops : Bus stop reference DataFrame (~19 k rows)
    """
    R = 6_371_000.0  # Earth radius in metres

    def to_cartesian(lat_deg: np.ndarray, lon_deg: np.ndarray) -> np.ndarray:
        """Convert lat/lon (degrees) to ECEF Cartesian coordinates."""
        lat = np.radians(lat_deg)
        lon = np.radians(lon_deg)
        x = R * np.cos(lat) * np.cos(lon)
        y = R * np.cos(lat) * np.sin(lon)
        z = R * np.sin(lat)
        return np.column_stack((x, y, z))

    # --- build KDTree from stop Cartesian coords (done once) ---
    stop_lats = df_stops[stop_lat_col].to_numpy()
    stop_lons = df_stops[stop_lon_col].to_numpy()
    stop_names = df_stops[stop_name_col].to_numpy()
    stop_ids = df_stops[stop_id_col].to_numpy()

    stop_cart = to_cartesian(stop_lats, stop_lons)  # (S, 3)
    tree = KDTree(stop_cart)

    # --- query all GPS points in one call ---
    gps_lats = df_bus[gps_lat_col].to_numpy()
    gps_lons = df_bus[gps_lon_col].to_numpy()
    gps_cart = to_cartesian(gps_lats, gps_lons)  # (N, 3)

    # workers=-1 uses all CPU cores
    euclidean_dists, indices = tree.query(gps_cart, workers=-1)

    # Convert Euclidean chord distance → great-circle arc distance (metres)
    # chord = 2R·sin(θ/2)  →  arc = R·θ = 2R·arcsin(chord / 2R)
    arc_dists = 2 * R * np.arcsin(np.clip(euclidean_dists / (2 * R), -1.0, 1.0))

    return df_bus.with_columns(
        [
            pl.Series(out_name_col, stop_names[indices].astype(str)),
            pl.Series(out_id_col, stop_ids[indices].astype(str)),
            pl.Series(out_dist_col, np.round(arc_dists, 2)),
        ]
    )

In [10]:
result = assign_nearest_stop(df_raw, df_stops)

In [39]:
result.filter(
    pl.col("RouteID") == 1728,
    pl.col("Direction") == "返程",
    pl.col("DutyStatus") == "開始",
    pl.col("NearestStopName") == "新竹轉運站",
).select(
    [
        "PlateNumb",
        "RouteID",
        "Direction",
        "PositionLon",
        "PositionLat",
        "GPSTime",
        "NearestStopName",
        "DistanceToStop_m",
    ]
).sort("GPSTime").select(pl.col("PlateNumb")).group_by("PlateNumb").len()

PlateNumb,len
str,u32
"""KKB-2035""",7
"""KKB-1713""",4
"""KKB-1715""",4
"""KKB-2058""",3
"""KKB-2669""",2
"""KKB-2056""",7
"""058-FS""",7
"""KKB-2059""",3


In [16]:
result.glimpse()

Rows: 736130
Columns: 29
$ PlateNumb         <str> '393-FL', '393-FL', '805-FZ', '393-FL', '393-FL', '393-FL', '393-FL', '393-FL', '393-FL', '393-FL'
$ OperatorID        <i64> 35, 35, 35, 35, 35, 35, 35, 35, 35, 35
$ OperatorNo        <i64> 1308, 1308, 1308, 1308, 1308, 1308, 1308, 1308, 1308, 1308
$ RouteUID          <str> 'THB0968', 'THB0968', 'THB0968', 'THB0968', 'THB0968', 'THB0968', 'THB0968', 'THB0968', 'THB0968', 'THB0968'
$ RouteID           <i64> 968, 968, 968, 968, 968, 968, 968, 968, 968, 968
$ RouteNameZh_tw    <i64> 968, 968, 968, 968, 968, 968, 968, 968, 968, 968
$ RouteNameEn       <i64> 968, 968, 968, 968, 968, 968, 968, 968, 968, 968
$ SubRouteUID       <str> 'THB096802', 'THB096802', 'THB096802', 'THB096801', 'THB096802', 'THB096801', 'THB096801', 'THB096801', 'THB096802', 'THB096802'
$ SubRouteID        <i64> 96802, 96802, 96802, 96801, 96802, 96801, 96801, 96801, 96802, 96802
$ SubRouteNameZh_tw <i64> 968, 968, 968, 968, 968, 968, 968, 968, 968, 968
$ SubRouteNameE

In [31]:
temp = pl.scan_parquet(DATA_FILE).filter(pl.col("RouteID") == 1728).pipe(clean_df)

In [38]:
temp.filter(
    pl.col("Direction") == "返程",
    pl.col("Stop") == "新竹轉運站",
    pl.col("Time").is_between(date(2025, 11, 23), datetime(2025, 11, 23, 23, 59, 59)),
).sort("Time").collect().select(pl.col("Plate").unique())

Plate
str
"""KKA-1832"""
"""056-FS"""
"""KKB-2057"""
"""KKB-1715"""
"""KKB-2056"""
"""KKB-2058"""
"""058-FS"""
"""057-FS"""
"""KKB-2059"""
